# Gefahrgut-Routenplanung & Fahrzeugzuweisung (PuLP)

---

## Problembeschreibung

Gegeben sind:

- Eine Menge **LKWs** mit individuellen Kapazitäts-, Reichweiten- und Kostenprofilen
- Eine Menge **Gefahrgutlieferungen**, jeweils mit Gewicht, Gefahrgutklasse, Startknoten und Zielknoten
- Ein **Straßennetz** modelliert als gerichteter Graph aus Knoten und Kanten
- **Risikodaten** je Kante, aggregiert aus Bevölkerungsdichte, Unfallrate und Naturschutznähe
- **Rechtliche Einschränkungen** je Kante und Gefahrgutklasse (ADR-Verordnung)
- **Ladeinfrastruktur** an ausgewählten Knotenpunkten für den Langstreckentransport

Ziel ist die simultane Minimierung von **Transportrisiko** und **Transportkosten** durch folgende Entscheidungen:
1. Welches Fahrzeug wird welcher Lieferung zugewiesen?
2. Über welche Route wird jede Lieferung durch das Netz geführt?
3. An welchen Knoten muss das Fahrzeug nachgeladen werden?

---

## Mengen (Sets)

| Symbol | Bedeutung |
|--------|-----------|
| $V$ | Menge der verfügbaren Fahrzeuge (z. B. MAN\_eTGX, Volvo\_FH\_Electric) |
| $L$ | Menge der durchzuführenden Gefahrgutlieferungen |
| $N$ | Menge aller Knoten (Kreuzungen, Depot, Ziele) im Straßennetz |
| $E$ | Menge aller gerichteten Kanten (Straßenabschnitte); Kante $e = (i, j)$ verläuft von Knoten $i$ nach Knoten $j$ |
| $K$ | Menge der Gefahrgutklassen (ADR-Klassifikation) |

---

## Parameter

| Symbol | Einheit | Bedeutung |
|--------|---------|-----------|
| $Cap_v$ | t | Maximale Nutzlast von Fahrzeug $v$ |
| $FC_v$ | €/Tag | Tägliche Fixkosten für den Einsatz von Fahrzeug $v$ |
| $VC_{v,e}$ | € | Variable Kosten für Fahrzeug $v$ auf Kante $e$ (km-Kosten + Energiekosten) |
| $Dem_l$ | t | Gewicht der Lieferung $l$ |
| $Class_l$ | — | Gefahrgutklasse $k \in K$ der Lieferung $l$ |
| $O_l,\, D_l$ | — | Start- und Zielknoten der Lieferung $l$ |
| $Risk_{e,k}$ | — | Risikoscore für Kante $e$ und Gefahrgutklasse $k$, normiert auf $[0, 1]$ |
| $Allow_{e,k}$ | $\{0,1\}$ | 1, wenn Kante $e$ für Gefahrgutklasse $k$ rechtlich freigegeben ist, sonst 0 |
| $Dist_e$ | km | Länge der Kante $e$ |
| $Range_v$ | km | Akkureichweite von Fahrzeug $v$ |
| $ChargeNode_n$| $\{0,1\}$ | 1, wenn an Knoten $n$ eine kompatible Ladesäule (HPC/DC) existiert, sonst 0 |

### Zusammensetzung des Risikoscores

Der Risikoscore je Kante aggregiert drei Datenquellen:

$$
Risk_{e,k} = \underbrace{\alpha \cdot \text{PopDens}_e}_{\text{Bevölkerungsdichte}}
           + \underbrace{\beta  \cdot \text{AccRate}_e}_{\text{Unfallrate}}
           + \underbrace{\gamma \cdot \text{NatRes}_e}_{\text{Naturschutznähe}}
\qquad \text{mit } \alpha + \beta + \gamma = 1
$$

- $\text{PopDens}_e$: aus `density.csv` (JRC Census Raster, 100 m Auflösung)
- $\text{AccRate}_e$: aus `Accidents.csv` (268 Tsd. Unfallereignisse → Spatial Join auf Kanten)
- $\text{NatRes}_e$: aus `nature_reserves_filtered.csv` (Naturschutzgebiete NW/NI)

### Zusammensetzung der variablen Kosten

$$
VC_{v,e} = Dist_e \cdot \underbrace{c^{km}_v}_{\text{Kosten pro km}}
         + Dist_e \cdot \underbrace{\varepsilon_v}_{\text{kWh/km}} \cdot \underbrace{p_e}_{\text{Energiepreis}}
$$

Der Energiepreis $p_e$ hängt vom Kantentyp ab: Depot-Laden (0,35 €/kWh), Stadtladen DC (0,60 €/kWh) oder Autobahn HPC (0,75 €/kWh).

---

## Entscheidungsvariablen

Wir verwenden eine **entkoppelte Formulierung**, die Routing und Fahrzeugzuweisung trennt. Gegenüber der naiven Formulierung $x_{l,v,e}$ reduziert sich die Anzahl der Binärvariablen um den Faktor $|V|$.

| Variable | Typ | Bedeutung |
|----------|-----|-----------|
| $f_{l,e} \in \{0,1\}$ | Binär | 1, wenn Lieferung $l$ über Kante $e$ transportiert wird |
| $y_{l,v} \in \{0,1\}$ | Binär | 1, wenn Lieferung $l$ dem Fahrzeug $v$ zugewiesen wird |
| $z_v \in \{0,1\}$ | Binär | 1, wenn Fahrzeug $v$ heute eingesetzt wird |
| $u_{l,n} \geq 0$ | Stetig | MTZ-Hilfsvariable, kodiert Besuchsreihenfolge an Knoten $n$ für Lieferung $l$ |
| $q_{l,n} \geq 0$ | Stetig | Verbleibende Restreichweite (in km) des Fahrzeugs für Lieferung $l$ bei Ankunft an Knoten $n$ |
| $c_{l,n} \in \{0,1\}$ | Binär | 1, wenn das Fahrzeug für Lieferung $l$ an Knoten $n$ aufgeladen wird |

**Warum entkoppelt?**

$$
\underbrace{|L| \times |V| \times |E|}_{\text{naiv: } x_{l,v,e}}
\quad \longrightarrow \quad
\underbrace{|L| \times |E|}_{\text{entkoppelt: } f_{l,e}} + \underbrace{|L| \times |V|}_{y_{l,v}}
$$

Bei 20 Lieferungen, 3 Fahrzeugen und 5.000 Kanten reduziert sich die Variablenanzahl von 300.000 auf 100.060 Binärvariablen.

---

## Zielfunktion

Wir minimieren eine **normierte gewichtete Summe** aus Risiko und Kosten:

$$
\min \quad
\frac{w_1}{Risk_{\max}} \sum_{l \in L} \sum_{e \in E} Risk_{e,\, Class_l} \cdot f_{l,e}
\;+\;
\frac{w_2}{Cost_{\max}} \left(
  \sum_{v \in V} FC_v \cdot z_v
  \;+\;
  \sum_{l \in L} \sum_{v \in V} \sum_{e \in E} VC_{v,e} \cdot y_{l,v} \cdot \frac{1}{|V|}
\right)
$$

**Warum normieren?** Risikoscores liegen in $[0, 1]$, Kosten hingegen in €, typischerweise um mehrere Größenordnungen größer. Ohne Normierung würde der Solver faktisch nur die Kosten optimieren. Durch Division jedes Terms durch sein theoretisches Maximum ($Risk_{\max}$ bzw. $Cost_{\max}$) werden beide auf dieselbe Skala gebracht. Mit $w_1 = w_2 = 0{,}5$ werden Risiko und Kosten gleichgewichtet — die Gewichte können für Sensitivitätsanalysen variiert werden.

---

## Nebenbedingungen

| Nr. | Name | Formel | Zweck |
|-----|------|--------|-------|
| C1 | Transportpflicht | $\sum_{v \in V} y_{l,v} = 1 \quad \forall l$ | Jede Lieferung genau einem Fahrzeug zugewiesen |
| C2 | Kapazität | $\sum_{l} Dem_l \cdot y_{l,v} \leq Cap_v \cdot z_v \quad \forall v$ | Nutzlastgrenze je Fahrzeug |
| C3 | Aktivierung | $z_v \geq y_{l,v} \quad \forall l, v$ | Fahrzeug muss aktiv sein, wenn Lieferung zugewiesen |
| C4 | Gefahrgutrestriktion | $f_{l,e} \leq Allow_{e,\, Class_l} \quad \forall l, e$ | Nur rechtlich erlaubte Routen |
| C5 | Flusserhaltung | siehe unten | Physikalisch konsistente, zusammenhängende Route |
| C6 | Subtour-Eliminierung | MTZ-Ungleichung | Keine isolierten Schleifen |
| C7 | Ladezustand (SoC) | $q_{l,j} \leq q_{l,i} - Dist_{(i,j)} + M(1 - f_{l,(i,j)}) + M c_{l,i}$ | Fortlaufende Erfassung der Restreichweite (EVRP) |
| C8 | Akkukapazität | $q_{l,n} \leq \sum_v Range_v \cdot y_{l,v} \quad \forall l, n$ | Restreichweite darf maximale Reichweite des Trucks nicht überschreiten |
| C9 | Ladeinfrastruktur | $c_{l,n} \leq ChargeNode_n \quad \forall l, n$ | Ladevorgänge sind nur an Knoten mit vorhandener Infrastruktur erlaubt |

### C1 Transportpflicht

Jede Lieferung muss exakt einem Fahrzeug zugewiesen werden also nicht null, nicht zwei:

$$
\sum_{v \in V} y_{l,v} = 1 \qquad \forall\, l \in L
$$

### C2 Kapazität

Das Gesamtgewicht aller einem Fahrzeug $v$ zugewiesenen Lieferungen darf dessen Nutzlast nicht überschreiten. Ist $z_v = 0$, erzwingt die rechte Seite den Wert 0 und verhindert jede Zuweisung:

$$
\sum_{l \in L} Dem_l \cdot y_{l,v} \leq Cap_v \cdot z_v \qquad \forall\, v \in V
$$

### C3 Aktivierung

Wird Fahrzeug $v$ eine Lieferung zugewiesen, muss es als aktiv markiert sein. Ohne diesen Constraint könnte C2 umgangen werden ($z_v = 0$ während $y_{l,v} = 1$):

$$
z_v \geq y_{l,v} \qquad \forall\, l \in L,\; v \in V
$$

### C4 Gefahrgutrestriktion

Eine Kante darf nur befahren werden, wenn sie für die Gefahrgutklasse der Lieferung rechtlich freigegeben ist. Implementierungshinweis: Constraints werden nur für explizit gesperrte Kanten hinzugefügt, was effizienter ist als eine Iteration über alle Kanten:

$$
f_{l,e} \leq Allow_{e,\, Class_l} \qquad \forall\, l \in L,\; e \in E
$$

### C5 Flusserhaltung

An jedem Knoten $n$ muss der eingehende Fluss dem ausgehenden entsprechen, außer am Startknoten (Nettoabfluss 1) und Zielknoten (Nettozufluss 1). Dies garantiert einen zusammenhängenden, lückenlosen Pfad von $O_l$ nach $D_l$:

$$
\sum_{e:\, e_1 = n} f_{l,e} - \sum_{e:\, e_0 = n} f_{l,e}
=
\begin{cases}
-1 & \text{falls } n = O_l \quad \text{(Quelle)} \\
+1 & \text{falls } n = D_l \quad \text{(Senke)} \\
\;\;0 & \text{sonst} \quad \text{(Durchgangsknoten)}
\end{cases}
\qquad \forall\, l \in L,\; n \in N
$$

### C6 Subtour-Eliminierung (Miller–Tucker–Zemlin)

Flusserhaltung allein verhindert keine Subtouren — der Solver könnte isolierte Schleifen zurückgeben (z. B. $A \to B \to A$ und $C \to D \to C$), die alle Flussgleichungen erfüllen, aber physikalisch sinnlos sind. Die MTZ-Constraints verwenden Hilfsvariablen $u_{l,n}$ zur Kodierung einer Besuchsreihenfolge, um dies zu verhindern:

$$
u_{l,i} - u_{l,j} + |N| \cdot f_{l,(i,j)} \leq |N| - 1
\qquad \forall\, l \in L,\; (i,j) \in E,\; j \neq O_l
$$

Wird Kante $(i, j)$ genutzt ($f = 1$), gilt $u_{l,j} > u_{l,i}$: Knoten $j$ muss strikt nach Knoten $i$ besucht werden. Eine Schleife würde erfordern, dass ein Knoten sowohl vor als auch nach sich selbst erscheint — ein Widerspruch.

Der Big-M-Wert wird auf $M = |N|$ gesetzt, was die engste mathematisch korrekte Schranke ist und numerische Stabilität gewährleistet.

### C7 Ladezustand (SoC) und C8 Akkukapazität

Um deutschlandweite Routen (die die Standardreichweite der E-Trucks übersteigen) zu ermöglichen, wird der Ladezustand (bzw. die Restreichweite) an jedem Knoten getrackt. Eine globale Summierung der Distanzen würde Langstrecken unmöglich machen. Die Big-M-Formulierung koppelt die Restreichweite an die gefahrene Kante und eventuelle Ladevorgänge:

$$
q_{l,j} \leq q_{l,i} - Dist_{(i,j)} + M \cdot (1 - f_{l,(i,j)}) + M \cdot c_{l,i} \qquad \forall\, l \in L,\; (i,j) \in E
$$

Wird die Kante $(i,j)$ befahren ($f=1$) und nicht geladen ($c=0$), sinkt die Restreichweite exakt um die Kantendistanz. Der Ladevorgang ($c=1$) "resettet" die Gleichung durch das Big-M. Gleichzeitig stellt C8 sicher, dass die getrackte Reichweite niemals die physikalische Akkukapazität des gewählten Trucks überschreitet:

$$
q_{l,n} \leq \sum_{v \in V} Range_v \cdot y_{l,v} \qquad \forall\, l \in L,\; n \in N
$$

### C9 Ladeinfrastruktur

Nachladen ist nur möglich, wenn die entsprechende physikalische Infrastruktur (z. B. ein HPC-Ladepark) an Knoten $n$ vorhanden ist:

$$
c_{l,n} \leq ChargeNode_n \qquad \forall\, l \in L,\; n \in N
$$

---

## Modelltyp

Dieses Modell ist ein:

- **Gemischt-ganzzahliges lineares Programm (MILP)** — binäre und stetige Variablen, lineare Zielfunktion und Nebenbedingungen
- **Electric Vehicle Routing Problem (EVRP)** — Multi-Commodity-Netzwerkfluss mit expliziter SoC-Nachverfolgung, Ladeinfrastruktur, Kapazitäten und Gefahrgutrecht
- Gelöst mit PuLPs Standard-**CBC-Solver** (Open Source); für größere Instanzen empfiehlt sich Gurobi oder CPLEX

---

## Ausgabe

Das Modell liefert:

- Den **Solver-Status** (Optimal / Infeasible / …)
- Welches **Fahrzeug welcher Lieferung zugewiesen** wird ($y_{l,v} = 1$)
- Welche **Kanten je Lieferung genutzt** werden ($f_{l,e} = 1$)
- An welchen **Knoten Ladevorgänge stattfinden** ($c_{l,n} = 1$)
- Welche **Fahrzeuge eingesetzt** werden ($z_v = 1$)
- Den **normierten Zielfunktionswert** sowie eine Kostenaufschlüsselung

---